# Welcome to Week 5 - Agent Frameworks

## Day 1: Google ADK (and a quick look at A2A)

This week is a fast tour of today's agent frameworks. The big idea is that once you know one of them well, you basically know them all. Every framework, we build an agent the same five steps, so the pattern starts to feel familiar fast:

1. **Create the agent** - give it a model and a system prompt.
2. **Run it** - send a message, get a reply.
3. **Add tools** - plain typed functions the agent can call.
4. **Add MCP** - connect a tool server someone else wrote, wired the same way every time.
5. **Put it in a loop with a goal** - hand it an objective and let it work, step after step, until the job is done.

Steps 1 and 2 are still just an LLM call. Tools and MCP give the agent things it can do. Step 5 is where it becomes an agent: the framework runs the loop on its own, choosing a tool, reading the result, choosing again, until the goal is met. That loop is the thing to watch all week.

We open with Google ADK, the framework we spend the most time on. It has the most polished developer experience of the bunch, a visual trace you can watch live, and first-class support for both MCP and A2A.

Our running project for the whole week is a small SQLite todo board. A worker picks one task off the board, does it with its agent loop, writes the result, and marks the task done. You build and run the whole worker right here, then see it again in ADK's visual UI. On Day 5 the same board grows up to coordinate a whole team of agents.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The Google ADK docs</h2>
            <span style="color:#00bfff;">The documentation lives at <a href="https://adk.dev">https://adk.dev</a> and is worth a browse. ADK ships two release lines, 1.x and 2.x; we use the latest 2.x throughout. Watch out for older blog posts, because several import paths changed recently.</span>
        </td>
    </tr>
</table>

## Setup

Two things this day needs, both already in place from earlier weeks:

- **Node**, for `npx` (the filesystem MCP server runs over it). Check with `node --version`.
- **`GOOGLE_API_KEY`** in the repo-root `.env`. You set this up in the CrewAI week, so it is already there.

Google ADK is part of the repo's environment, so a normal `uv sync` from the repo root installs everything. Open this notebook in Cursor and pick the repo's default **Python 3.12.12** kernel, the same one you use every week, then run the cells top to bottom.

The notebook starts the filesystem MCP server for you at step 4, over `npx`. Warm it once now so that first run is fast, then stop it with Ctrl-C as soon as it prints that it is running:

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from quiet import silence
silence() ## Turn down some noisy library logging so the agent's trace is easy to read

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI","FALSE")
load_dotenv(override=True)

True

## Step 1: Create the agent

In ADK an agent is an `LlmAgent`: a model plus an `instruction`, which is its system prompt. That is the whole object, no wrapper class and no registration step. First the setup; `GOOGLE_API_KEY` comes from the repo-root `.env`.

In [3]:
Model = LiteLlm(model="groq/openai/gpt-oss-120b")

Agent = LlmAgent(
    model=Model,
    name="Assistant",
    instruction="You are a concise, friendly assistant. Reply in a single short sentence"
)

## Step 2: Run it

`InMemoryRunner(...).run_debug(..., verbose=True)` runs the agent and prints each step, so you can watch what it does. With no tools yet there is nothing to loop over, so the agent just replies. This is still only an LLM call.

In [4]:
Result = await InMemoryRunner(agent=Agent).run_debug("Say hello in Spanish", verbose=True)

Assistant > The user asks: "Say hello in Spanish". Need to reply in a single short sentence, as per developer instruction: concise, friendly, single short sentence. So answer: "¡Hola!" That's a short sentence.¡Hola!


## Our project this week: a SQLite todo board

The worker coordinates through a tiny SQLite board: one file, one table, no server to run. `board.py` is a handful of small functions. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [5]:
import board
board.ResetBoard()
board.AddGoal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt")
board.ListTodos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [7]:
board.ShowBoard()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt

## Step 3: Add tools

A tool in ADK is just a typed Python function with a docstring. The docstring becomes the description the model reads, the type hints become the argument schema, and you hand the function to the agent in its `tools=[...]` list.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [8]:
def ShowTodos() -> list[dict]:
    """List every Todo on the board. A goal has parent_id None.
    A step has parent_id set to its goal's id"""
    return board.ListTodos()

def PlanSteps(GoalId: int, Steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id
    and a short list of step descriptions"""
    return {"goal_id":GoalId, "step_ids": [board.AddStep(GoalId,Step) for Step in Steps]}

def CompleteTask(TaskId: int, Result: str) -> dict:
    """Mark a Todo (goal or a step) as done and record a short result summary"""
    board.CompleteTodo(TaskId,Result)
    return {"task_id": TaskId, "status":"done"}

In [11]:
BoardAgent = LlmAgent(
    model=LiteLlm("openai/gpt-5.4-nano"),
    name="BoardAgent",
    instruction="You help manage a shared Todo board",
    tools = [ShowTodos, CompleteTask]
)

In [12]:
Result = await InMemoryRunner(agent=BoardAgent).run_debug("What is on the board right now, and what is its status?", verbose=True)

BoardAgent > [Calling tool: ShowTodos({})]
BoardAgent > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'Read notes.txt, translate its contents into natura...]
BoardAgent > Right now the board has **1 todo**:

- **(Goal #1)**: “Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt”  
  - **Status:** pending  
  - **Result:** (none yet)


### A tool can reach the real world too

The board tools talk to a little database. A tool can just as easily reach outside the machine. Here is one that sends a push notification to your phone through Pushover. It is the same `send_push_notification` used elsewhere in the course, and it is still just a typed function with a docstring; ADK reads those to learn how to call it.

In [16]:
import requests

def SendPushNotification(Message: str) -> str:
    """Send a short push notification to report that a job is done or to report a result"""
    Topic = "ross919gain370"
    URL = f"https://ntfy.sh/{Topic}"
    Response = requests.post(URL, data=Message.encode(encoding="utf-8"))
    return f"Push notification sent with API return code {Response.status_code}"

In [17]:
Notifier = LlmAgent(
    model = "gemini-3.1-flash-lite",
    name="Notifier",
    instruction="""You notify the user when asked. Use your SendPushNotification tool to send a push notification
     and use the ShowTodos tool to get the state of the board. Limit the notification to a couple of sentences
    """,
    tools = [ShowTodos, SendPushNotification]
)

Result = await InMemoryRunner(agent=Notifier).run_debug("Send a push notification with the state of the board", verbose=True)

Notifier > [Calling tool: ShowTodos({})]
Notifier > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'Read notes.txt, translate its contents into natura...]
Notifier > [Calling tool: SendPushNotification({'Message': 'The current board contains one pendin...)]
Notifier > [Tool result: {'result': 'Push notification sent with API return code 200'}]
Notifier > The state of the board has been retrieved, and a push notification has been sent informing you that there is currently one pending task: "Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt."


## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server (the same Node server every framework uses this week), scoped to a single `workspace` folder so it can only touch files in there. In ADK that is one `McpToolset` added to the same `tools=[...]` list.

ADK starts the server for you as a small Node process over `npx`. One detail worth knowing: we pass `errlog=subprocess.DEVNULL`. That quiets the server's startup logging, and it is also what lets the server run from a Jupyter kernel on Windows, where the kernel's stderr has no real file descriptor for the server to write to. On Mac and Linux it changes nothing.

In [21]:
import subprocess
from pathlib import Path
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

Workspace = Path("task_worker/workspace").resolve() ## The only folder the agent can access

Filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params = StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(Workspace)],
            cwd = str(Workspace) ## Start the server in the workspace so relative file names resolve there
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL
)

In [22]:
FileAgent = LlmAgent(
    model="gemini-3.1-flash-lite",
    name="FileAgent",
    instruction="You can read and write files in your workspace. Use your tools to do what is asked",
    tools = [Filesystem]
)

In [23]:
Result = await InMemoryRunner(agent=FileAgent).run_debug("Read notes.txt and summarize it one short sentence", verbose=True)

FileAgent > [Calling tool: list_allowed_directories({})]
FileAgent > [Tool result: {'content': [{'type': 'text', 'text': 'Allowed directories:\nD:\\Python\\Agents2\\5_agent_frameworks...]
FileAgent > [Calling tool: list_directory({'path': '.'})]
FileAgent > [Tool result: {'content': [{'type': 'text', 'text': '[FILE] notes.txt'}], 'structuredContent': {'content': '[FILE]...]
FileAgent > [Calling tool: read_text_file({'path': 'notes.txt'})]
FileAgent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\r\n\r\nToday we are building a small lan...]
FileAgent > The team is working together to build a small language tutor by carefully completing individual tasks from a shared board.


## Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat.

This time we drive the loop with `run_async` rather than `run_debug`, stepping through its events ourselves. We print each tool call as the agent makes it, and re-draw the board whenever a step is planned or ticked off, so you watch the plan appear and then strike through one line at a time.

In [24]:
from google.genai import types

INSTRUCTIONS = """
               You are a careful worker with a shared Todo board and a set of file tools.
               Take the pending goal and see it through. Begin by laying out a short plan.
               This plan comprises of the set of concrete steps the work breaks down into.
               Then put these steps into the board under the goal, and start working on these
               steps. Update the status of the steps using the tools you have. You also have a 
               tool to read and write files to a location. Once you have finished all the tasks,
               close the associated goal on the board and send a push notification to the user
               stating in 2-3 sentences what was accomplished.
               """

Worker = LlmAgent(
    model = "openai/gpt-5.4-nano",
    name = "TaskWorker",
    instruction = INSTRUCTIONS,
    tools = [ShowTodos,CompleteTask,PlanSteps,Filesystem,SendPushNotification]
)

board.ResetBoard()
GoalId = board.AddGoal(
    """Read notes.txt, translate its contents into natural Spanish, and write the Spanish
    to spanish.txt, and send a push notification to the user"""
)
board.ClaimTodo(GoalId)

## Drive the loop ourselves with run_async instead of run_debug, so that we can watch it
## print each tool call, and redraw the board whenever a step is planned or ticked off
Runner = InMemoryRunner(agent=Worker)
Session = await Runner.session_service.create_session(app_name=Runner.app_name, user_id="Siddharth")

async for Event in Runner.run_async(
    user_id="Siddharth",
    session_id = Session.id,
    new_message = types.UserContent("Please work the pending goal on the board")
):
    for Call in Event.get_function_calls():
        print (f"Tool: {Call.name}({Call.args})")
    if any(r.name in ("PlanSteps","CompleteTask") for r in Event.get_function_responses()):
        board.ShowBoard()
    if Event.is_final_response():
        print (Event.content.parts[0].text)

Tool: ShowTodos({})
Tool: PlanSteps({'GoalId': 1, 'Steps': ['Locate notes.txt in the allowed workspace and read its contents', 'Translate the contents into natural Spanish', 'Write the translation to spanish.txt', 'Complete the goal and send a brief push notification summarizing the work']})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish
    to spanish.txt, and send a push notification to the user
  Step #2: Locate notes.txt in the allowed workspace and read its contents
  Step #3: Translate the contents into natural Spanish
  Step #4: Write the translation to spanish.txt
  Step #5: Complete the goal and send a brief push notification summarizing the work

Tool: list_allowed_directories({})
Tool: search_files({'path': 'D:\\Python\\Agents2\\5_agent_frameworks\\1_google_adk_a2a\\task_worker\\workspace', 'pattern': '**/notes.txt'})
Tool: read_text_file({'path': 'D:\\Python\\Agents2\\5_agent_frameworks\\1_google_adk_a2a\\task_worker\\workspace\\notes.txt'})
Tool: edit_file({'path': 'D:\\Python\\Agents2\\5_agent_frameworks\\1_google_adk_a2a\\task_worker\\workspace\\spanish.txt', 'edits': [{'oldText': '', 'newText': 'Bienvenido al equipo.\n\nHoy estamos construyendo un pequeño tutor de idiomas, pieza por pieza. Cada ayudante\nelige una sola tarea de la pizarra compartida, realiza el trabajo con cuidado y marca\nla tarea como completada. Lee tu tarea con atención, tómate tu tiempo y haz\nlo mejor que puedas.\n'}]})
Tool: write_file({'path': 'D:\\Python\\Agents2\\5_agent_frameworks\\1_google_adk_a2a\\task_worker\\workspace\\spanish.txt', 'content': 'Bienvenido al equipo.\n\nHoy estamos construyendo un pequeño tutor de idiomas, pieza por pieza. C

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish
    to spanish.txt, and send a push notification to the user
  Step #2: Locate notes.txt in the allowed workspace and read its contents  Found and read notes.txt from the workspace.
  Step #3: Translate the contents into natural Spanish
  Step #4: Write the translation to spanish.txt
  Step #5: Complete the goal and send a brief push notification summarizing the work

Tool: CompleteTask({'TaskId': 3, 'Result': 'Translated the English notes into natural Spanish.'})


...

Tool: CompleteTask({'TaskId': 5, 'Result': 'Completed goal: notes.txt translated to Spanish in spanish.txt.'})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish
    to spanish.txt, and send a push notification to the user
  Step #2: Locate notes.txt in the allowed workspace and read its contents  Found and read notes.txt from the workspace.
  Step #3: Translate the contents into natural Spanish  Translated the English notes into natural Spanish.
  Step #4: Write the translation to spanish.txt  Wrote the Spanish translation to spanish.txt in the workspace.
  Step #5: Complete the goal and send a brief push notification summarizing the work  Completed goal: notes.txt translated to Spanish in spanish.txt.

Tool: CompleteTask({'TaskId': 1, 'Result': 'Read notes.txt, translated it into natural Spanish, and wrote the result to spanish.txt.'})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish
    to spanish.txt, and send a push notification to the user  Read notes.txt, translated it into natural Spanish, and wrote the result to spanish.txt.
  Step #2: Locate notes.txt in the allowed workspace and read its contents  Found and read notes.txt from the workspace.
  Step #3: Translate the contents into natural Spanish  Translated the English notes into natural Spanish.
  Step #4: Write the translation to spanish.txt  Wrote the Spanish translation to spanish.txt in the workspace.
  Step #5: Complete the goal and send a brief push notification summarizing the work  Completed goal: notes.txt translated to Spanish in spanish.txt.

Tool: SendPushNotification({'Message': 'Completé la tarea: leí notes.txt, lo traduje a español de forma natural y guardé el resultado en spanish.txt.\nSi deseas, puedo revisar el texto para ajustarlo a un estilo (más formal o más conversacional).'})
Plan:
1) Find and read `notes.txt` from the workspace.  
2) Translate its contents into natural Spanish.  
3) Write the translation to `spanish.txt`.  
4) Mark steps and the goal complete, then send a push notification.

Completed:
- Read `notes.txt`
- Translated it into natural Spanish
- Wrote the translation to `spanish.txt`
- Closed the goal and sent a push notification summarizing the work


In [2]:
from task_worker.agent import root_agent
import board
from google.genai import types

board.ResetBoard()
GoalId = board.AddGoal(
    """Read notes.txt, translate its contents into natural Hindi, and write the Hindi
    to hindi.txt, and send a push notification to the user"""
)
board.ClaimTodo(GoalId)

Runner = InMemoryRunner(agent=root_agent)
Session = await Runner.session_service.create_session(app_name=Runner.app_name, user_id="Siddharth")

async for Event in Runner.run_async(
    user_id="Siddharth",
    session_id = Session.id,
    new_message = types.UserContent("Please work the pending goal on the board")
):
    for Call in Event.get_function_calls():
        print (f"Tool: {Call.name}({Call.args})")
    if any(r.name in ("PlanSteps","CompleteTask") for r in Event.get_function_responses()):
        board.ShowBoard()
    if Event.is_final_response():
        print (Event.content.parts[0].text)

Tool: ShowTodos({})
Tool: PlanSteps({'Goalid': 1, 'Steps': ['Read notes.txt from the working directory', 'Translate the English contents into natural Hindi', 'Write the translation to hindi.txt', 'Mark translation steps complete and close the goal; send push notification']})


Goal #1: Read notes.txt, translate its contents into natural Hindi, and write the Hindi
    to hindi.txt, and send a push notification to the user

Tool: PlanSteps({'Goalid': 1, 'Steps': ['Read notes.txt from the working directory', 'Translate the English contents into natural Hindi', 'Write the translation to hindi.txt', 'Mark translation steps complete and close the goal; send push notification']})


Goal #1: Read notes.txt, translate its contents into natural Hindi, and write the Hindi
    to hindi.txt, and send a push notification to the user

Tool: PlanSteps({'GoalId': 1, 'Steps': ['Read notes.txt from the working directory', 'Translate the English contents into natural Hindi', 'Write the translation to hindi.txt', 'Mark translation steps complete and close the goal; send push notification']})


...

Tool: CompleteTask({'TaskId': 5, 'Result': 'All translation steps completed; goal ready to close.'})


Goal #1: Read notes.txt, translate its contents into natural Hindi, and write the Hindi
    to hindi.txt, and send a push notification to the user
  Step #2: Read notes.txt from the working directory  Read notes.txt and extracted the English text to translate.
  Step #3: Translate the English contents into natural Hindi  Translated the notes into natural Hindi with clear, readable phrasing.
  Step #4: Write the translation to hindi.txt  Wrote the Hindi translation to hindi.txt.
  Step #5: Mark translation steps complete and close the goal; send push notification  All translation steps completed; goal ready to close.

Tool: CompleteTask({'TaskId': 1, 'Result': 'Translated notes.txt into natural Hindi and wrote it to hindi.txt.'})


Goal #1: Read notes.txt, translate its contents into natural Hindi, and write the Hindi
    to hindi.txt, and send a push notification to the user  Translated notes.txt into natural Hindi and wrote it to hindi.txt.
  Step #2: Read notes.txt from the working directory  Read notes.txt and extracted the English text to translate.
  Step #3: Translate the English contents into natural Hindi  Translated the notes into natural Hindi with clear, readable phrasing.
  Step #4: Write the translation to hindi.txt  Wrote the Hindi translation to hindi.txt.
  Step #5: Mark translation steps complete and close the goal; send push notification  All translation steps completed; goal ready to close.

Tool: SendPushNotification({'Message': 'notes.txt की सामग्री को पढ़कर उसे स्वाभाविक हिंदी में अनुवादित किया और वही अनुवाद hindi.txt में लिख दिया। अनुवाद प्रक्रिया पूरी करके बोर्ड का संबंधित काम पूरा कर दिया गया है।'})
योजना:
1) `notes.txt` पढ़ना  
2) उसकी सामग्री का स्वाभाविक हिंदी में अनुवाद करना  
3) अनुवाद को `hindi.txt` में लिखना  
4) बोर्ड पर स्टेप्स/गोल को पूरा मार्क करना और पुश नोटिफिकेशन भेजना  

पूरा किया गया:
- `notes.txt` पढ़कर उसका स्वाभाविक हिंदी अनुवाद बनाया
- अनुवाद `hindi.txt` में लिख दिया
- बोर्ड पर सभी संबंधित स्टेप्स और मुख्य गोल “done” कर दिए
- पुश नोटिफिकेशन भेज दी गई (कार्य पूरा होने की पुष्टि के साथ)


## Which Runner, and which run method?

Worth a moment, because you meet both in every ADK project.

**The Runner.** We used `InMemoryRunner`: a `Runner` whose session, artifact and memory services live in memory. Ideal for learning and experiments, since nothing is saved and it all resets when the kernel stops. The plain `Runner` is what a real app uses. You give it the same agent plus real services, say a database-backed session service, so conversations persist across runs and across users.

**The run method.** Steps 2 to 4 used `run_debug`: a convenience that creates a session, sends your message, prints the trace, and returns the list of events. ADK's own docs call it debugging and experimentation only. Step 5 stepped up to `run_async`, the real entry point, where you own the session and iterate the event stream yourself. That is what let us print each tool call and re-draw the board as the loop ran:

```python
from google.genai import types

runner = InMemoryRunner(agent=agent)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ed")
message = types.UserContent("Say hi in French.")

async for event in runner.run_async(user_id="ed", session_id=session.id, new_message=message):
    print(event.author, event.content)
```

`run_debug` hides all of that so you can focus on the agent; `run_async` hands it back when you want the control, as in step 5. There is also `run`, a plain synchronous version of `run_async` for code that is not async, and `run_live`, an experimental streaming mode for real-time audio and video.

## The same worker, as an ADK project

You just built the whole worker here in the notebook. The `task_worker/` folder is that exact thing packaged as a module: an `__init__.py` and an `agent.py` holding the same board tools, the same filesystem server, and the same instruction. Packaging it as a folder is what lets ADK's tooling load it. An ADK agent is just a folder like this, and you can scaffold a fresh one from scratch with `adk create <name>`.

You get two things for free once it is a module. First, run it as a plain script, which seeds a goal and runs the loop exactly as above. Open a terminal in this day's folder and run:

```bash
uv run worker.py
```

Second, and the reason ADK is the nicest developer experience of the week, watch it in the live visual trace:

```bash
uv run worker.py --seed-only   # put a fresh goal on the board
uv run adk web                 # then open the link it prints
```

Pick **task_worker** from the dropdown (it also lists `a2a_demo`, which is for the A2A section below, so ignore it here) and type "please work the pending goal on the board". Every tool call and MCP call lights up in the timeline as the loop runs, the same loop you just watched here, now with a UI.

## A quick look at A2A

A2A (Agent2Agent) is a standard for letting separate agents discover and call each other over HTTP. ADK supports it in two calls: `to_a2a` exposes an agent as a service, and `RemoteA2aAgent` calls a remote one as if it were a local sub-agent. The `a2a_demo/` folder has a small example: a translator agent exposed as a service, and a local concierge that delegates translation to it.

First, if an `adk web` from the previous section is still running, stop it with Ctrl-C. This demo launches its own `adk web` from inside `a2a_demo`, and `adk web` only lists the agents in the folder you launch it from. Launching it there is what puts `spanish_concierge` in the dropdown; launched from the day root you would see `task_worker` instead.

Use two terminals, both inside the `a2a_demo` folder:

```bash
cd a2a_demo

# Terminal 1: expose the translator as an A2A service on port 8001
uv run uvicorn server:a2a_app --host localhost --port 8001

# Terminal 2: fetch the translator's agent card, then drive the concierge
curl http://localhost:8001/.well-known/agent-card.json
uv run adk web   # pick spanish_concierge, then ask it to translate something into Spanish
```

How does the concierge find the translator? We told it where to look. In `spanish_concierge/agent.py` the `RemoteA2aAgent` is built with the translator's card URL, `http://localhost:8001/.well-known/agent-card.json`. There is no broadcast or auto-scan; an A2A client is simply pointed at a known card URL. The `curl` above fetches that very card, which is how the concierge learns the agent's name, what it can do, and where to reach it before sending a single request. The concierge then delegates translation over A2A, and the Spanish comes back from the other process.

A2A is a real Linux Foundation standard and the discovery model is clean, but the hype runs well ahead of real adoption, and MCP is what carries day-to-day agent work right now. A2A does not come back in the Day 5 project.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Seed a different goal on the board, for example "write a short haiku about Madrid into madrid.txt", and run the worker again. Does it plan sensible steps and pick the right file tools? Then add a fourth tool of your own, a plain typed function, to the worker and watch it show up in the trace.</span>
        </td>
    </tr>
</table>